In [1]:
#!/usr/bin/env python3

import os, sys, math, random, datetime, json, gc, time
import numpy as np
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 프로젝트 경로(필요 시 수정)
current_dir = os.path.dirname(os.path.abspath(''))
two_up_dir = os.path.dirname(os.path.dirname(current_dir))
if two_up_dir not in sys.path:
    sys.path.append(two_up_dir)

# 모델/데이터셋/로스 임포트 (경로는 사용 환경에 맞게 구성되어 있다고 가정)
from TinyCenterSpeed.src.models.CenterSpeed import CenterSpeedDense, CenterSpeedDenseResidual, CenterSpeedBottleneckCBAM, CenterSpeedBottleneckSENet
from TinyCenterSpeed.dataset.CenterSpeed_dataset import CenterSpeedDataset, RandomRotation, RandomFlip
from TinyCenterSpeed.src.models.losses import *  # 필요 시 내부 함수 사용

# W&B 설정
use_wandb = True
# use_wandb = False  # W&B 사용 여부 설정
project_name = "CenterSpeedBottleneckCBAM_no_spatial"
run_name = "train_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    import wandb
    if use_wandb:
        if os.environ.get("WANDB_MODE","").lower() == "offline":
            wandb.init(project=project_name, name=run_name, mode="offline")
        else:
            try:
                wandb.init(project=project_name, name=run_name)
            except Exception:
                wandb.init(project=project_name, name=run_name, mode="offline")
except Exception as e:
    print(f"[wandb] 사용 불가: {type(e).__name__}: {e}")
    use_wandb = False
    wandb = None

# 재현성/디바이스
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.backends.cudnn.benchmark = True



wandb: Currently logged in as: whdaudpark (whdaudpark-dongguk-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Device: cuda


In [2]:

# ===== 경로 설정 =====
# 아래 4개 경로를 자신의 데이터 구조에 맞게 준비하세요.
train_obj_path  = "/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT"
train_free_path = "/home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT_free"
val_obj_path    = "/home/harry/sim_ws/src/f1tenth_gym_ros/Val_set"
val_free_path   = "/home/harry/sim_ws/src/f1tenth_gym_ros/Val_set_free"

# ===== 하이퍼파라미터 =====
image_size    = 128   # CenterSpeedDataset와 일치
pixelsize     = 0.1   # m/pixel
sigma_px      = 1.0   # 가우시안 σ (px)
epochs        = 100
batch_size    = 32
learning_rate = 5e-4

# DataLoader 최적화
NUM_WORKERS = min(os.cpu_count() or 4, 8)
PIN_MEMORY  = torch.cuda.is_available()
PERSIST     = NUM_WORKERS > 0

print({
    "image_size": image_size,
    "pixelsize": pixelsize,
    "sigma_px": sigma_px,
    "epochs": epochs,
    "batch_size": batch_size,
    "lr": learning_rate,
    "workers": NUM_WORKERS
})



# --- W&B 메트릭 정의/환경 기록 (LR 시각화를 위해 추가) ---
if 'wandb' in globals() and wandb and use_wandb:
    # epoch을 기준 스텝으로 사용
    try:
        wandb.define_metric("epoch")
        wandb.define_metric("lr", step_metric="epoch")
    except Exception:
        pass
    # 실험 하이퍼파라미터 기록
    try:
        wandb.config.update({
            "image_size": image_size,
            "pixelsize": pixelsize,
            "sigma_px": sigma_px,
            "epochs": epochs,
            "batch_size": batch_size,
            "lr_init": learning_rate,
            "optimizer": "Adam",
            "scheduler": "ReduceLROnPlateau(factor=0.5, patience=8, threshold=0.005)",
        }, allow_val_change=True)
    except Exception:
        pass



from torch.utils.data import ConcatDataset

# 객체 없음 데이터에서 GT를 0으로 강제하는 래퍼
class ZeroTargetWrapper(torch.utils.data.Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        inputs, gts, data_vec, dense_feats, is_free = self.base[idx]
        # heatmap, dense 모두 0으로 (객체 없음 학습)
        gts = torch.zeros_like(gts)
        dense_feats = torch.zeros_like(dense_feats)
        return inputs, gts, data_vec, dense_feats, is_free

# 변환 (원하면 활성화)
from torchvision import transforms as T
transform = T.Compose([
    RandomRotation(45, image_size=image_size),
    RandomFlip(0.5),
])

# 각 데이터셋 로드 (객체 있음/없음, train/val)
def make_dataset(root_dir, use_transform=False):
    ds = CenterSpeedDataset(dataset_path=root_dir, transform=(transform if use_transform else None), dense=True)
    # 픽셀/이미지/σ 설정
    try:
        ds.change_image_size(image_size)
    except Exception:
        ds.image_size = image_size
    try:
        ds.change_pixel_size(pixelsize)
    except Exception:
        ds.pixelsize = pixelsize
    ds.sx = sigma_px
    ds.sy = sigma_px
    return ds

# Train
train_obj_dataset  = make_dataset(train_obj_path, use_transform=False)
train_free_dataset = make_dataset(train_free_path, use_transform=False)
train_free_dataset = ZeroTargetWrapper(train_free_dataset)

train_dataset = ConcatDataset([train_obj_dataset, train_free_dataset])

# Val
val_obj_dataset  = make_dataset(val_obj_path, use_transform=False)
val_free_dataset = make_dataset(val_free_path, use_transform=False)
val_free_dataset = ZeroTargetWrapper(val_free_dataset)

val_dataset = ConcatDataset([val_obj_dataset, val_free_dataset])

print("train_obj:", len(train_obj_dataset), 
      "train_free:", len(train_free_dataset), 
      "=> train_total:", len(train_dataset))
print("val_obj:", len(val_obj_dataset), 
      "val_free:", len(val_free_dataset), 
      "=> val_total:", len(val_dataset))


# import os, glob

# def check_dir(root):
#     exists = os.path.isdir(root)
#     csvs   = sorted(glob.glob(os.path.join(root, "*.csv")))  # 재귀 미사용
#     print(f"[CHECK] {root}")
#     print("  exists:", exists)
#     print("  n_csv (non-recursive):", len(csvs))
#     for p in csvs[:5]:
#         print("   -", os.path.basename(p))
#     # 재귀로도 한 번 확인
#     all_csvs = sorted(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True))
#     print("  n_csv (recursive):", len(all_csvs))
#     print()

# check_dir(train_obj_path)
# check_dir(train_free_path)
# check_dir(val_obj_path)
# check_dir(val_free_path)






def worker_init_fn(_):
    try:
        import torch, os
        torch.set_num_threads(1)
        os.environ.setdefault("OMP_NUM_THREADS", "1")
        os.environ.setdefault("MKL_NUM_THREADS", "1")
    except Exception:
        pass

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSIST,
    prefetch_factor=4,
    drop_last=True,
    worker_init_fn=worker_init_fn,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSIST,
    prefetch_factor=2,
    drop_last=False,
    worker_init_fn=worker_init_fn,
)

len(train_loader), len(val_loader)


###################################  CenterSpeedDense & CenterSpeedDenseResidual ##########################################
# model = CenterSpeedDense(input_channels=4, image_size=image_size).to(device)

model = CenterSpeedBottleneckCBAM(input_channels=4, image_size=image_size, cbam_no_spatial=True).to(device)
# model = CenterSpeedDenseResidual(input_channels=4, image_size=image_size).to(device)

# model = CenterSpeedBottleneckSENet(input_channels=4, image_size=image_size).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=8,threshold=0.005)  # val loss 따라 LR 감소

# --- 현재 LR을 얻는 헬퍼 (추가) ---
def get_lr(optim):
    return optim.param_groups[0]["lr"]




{'image_size': 128, 'pixelsize': 0.1, 'sigma_px': 1.0, 'epochs': 100, 'batch_size': 32, 'lr': 0.0005, 'workers': 8}
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs1.csv
Entries     :  14576
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs2.csv
Entries     :  11293
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs3.csv
Entries     :  8009
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs4.csv
Entries     :  3813
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/1floor_obs5.csv
Entries     :  4428
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs1.csv
Entries     :  2361
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs10.csv
Entries     :  842
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielberg_map_obs2.csv
Entries     :  2143
Reading file:  /home/harry/sim_ws/src/f1tenth_gym_ros/Train_GT/Spielb

In [3]:

# # 출력 [B,4,H,W], gts [B,H,W], dense [B,H,W,3]
def dense_loss(output, gt_heatmap, gt_dense_data, is_free, alpha=0.99, decay=1.0):
    preds = output.permute(0,2,3,1)  # [B,H,W,4]
    w = gt_heatmap.unsqueeze(-1)     # [B,H,W,1]
    loss_occ   = (alpha     * (1 + w) * (preds[...,0:1] - gt_heatmap.unsqueeze(-1))**2).sum()
    loss_dense = ((1-alpha) * (1 + w) * (preds[...,1:]  - gt_dense_data)**2).sum()
    batch_size = output.shape[0]
    return (loss_occ + loss_dense) / batch_size

print("Model/optimizer/loss 준비 완료")


Model/optimizer/loss 준비 완료


In [ ]:

# import torch.nn.functional as F

# # heatmap의 출력에는 focal loss를 적용
# def dense_loss(output, gt_heatmap, gt_dense_data, is_free,focal_alpha = 0.25, alpha=0.95, gamma=2.0):
#     preds = output.permute(0,2,3,1)  # [B,H,W,4]
#     pred_occ   = preds[...,0]        # [B,H,W]
#     pred_dense = preds[...,1:]       # [B,H,W,3]

#     # --- focal loss for occupancy ---
#     # BCE with logits (sigmoid 안쓰면 여기서 sigmoid)
#     bce_loss = F.binary_cross_entropy_with_logits(pred_occ, gt_heatmap, reduction="none")
#     pt = torch.exp(-bce_loss)   # 확률 (예측이 맞을수록 1에 가까움)
#     focal_loss = (focal_alpha * (1-pt)**gamma * bce_loss).sum()

#     # --- dense regression loss (원래대로 MSE) ---
#     w = gt_heatmap.unsqueeze(-1)  # [B,H,W,1]
#     loss_dense = ((1-alpha) * (1 + w) * (pred_dense - gt_dense_data)**2).sum()

#     batch_size = output.shape[0]
#     return (focal_loss + loss_dense) / batch_size


In [4]:


best_val = float('inf')
train_hist, val_hist = [], []

save_dir = "/home/harry/ros2_ws/src/TinyCenterSpeed/src/pt"
os.makedirs(save_dir, exist_ok=True)

for epoch in range(1, epochs+1):
    model.train()
    running = 0.0
    for batch in train_loader:
        inputs, gts, data_vec, dense_feats, is_free = batch
        inputs      = inputs.to(device, non_blocking=True)
        gts         = gts.to(device, non_blocking=True)
        dense_feats = dense_feats.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        out = model(inputs)
        loss = dense_loss(out, gts, dense_feats, is_free, alpha=0.99)
        loss.backward()
        optimizer.step()
        running += loss.item()

    train_loss = running / max(1, len(train_loader))
    train_hist.append(train_loss)

    # Validation
    model.eval()
    v_running = 0.0
    with torch.no_grad():
        for batch in val_loader:
            inputs, gts, data_vec, dense_feats, is_free = batch
            inputs      = inputs.to(device, non_blocking=True)
            gts         = gts.to(device, non_blocking=True)
            dense_feats = dense_feats.to(device, non_blocking=True)
            out = model(inputs)
            v_loss = dense_loss(out, gts, dense_feats, is_free, alpha=0.99)
            v_running += v_loss.item()

    val_loss = v_running / max(1, len(val_loader)) if len(val_loader)>0 else train_loss
    val_hist.append(val_loss)

    # 스케줄러 업데이트
    scheduler.step(val_loss)

    # --- W&B 로그에 epoch 기준으로 LR까지 함께 로깅 (추가) ---
    if 'wandb' in globals() and wandb and use_wandb:
        try:
            wandb.log({
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "lr": get_lr(optimizer)
            }, step=epoch)
        except Exception:
            pass
    a = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # 로그
    msg = f"[{epoch:03d}/{epochs}] train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | lr={get_lr(optimizer):.6g}"
    print(msg)

    # 베스트 저장
    if val_loss < best_val:
        best_val = val_loss
        best_path = os.path.join(save_dir, f"CenterSpeedBottleneckCBAM_no_spatial{a}_epoch_{epoch}.pt")
        torch.save(model.state_dict(), best_path)
        print(f"  ↳ Best model saved: {best_path} (val={best_val:.6f})")

# 마지막 모델 저장
last_path = os.path.join(save_dir, f"CenterSpeedBottleneckCBAM_no_spatial{a}_epoch_{epoch}.pt")
torch.save(model.state_dict(), last_path)
print(f"Training finished. Model saved at {last_path}")


[001/100] train_loss=2022.815116 | val_loss=8.342976 | lr=0.0005
  ↳ Best model saved: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/CenterSpeedBottleneckCBAM_no_spatial20250821_214535_epoch_1.pt (val=8.342976)
[002/100] train_loss=2.657774 | val_loss=2.726663 | lr=0.0005
  ↳ Best model saved: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/CenterSpeedBottleneckCBAM_no_spatial20250821_214841_epoch_2.pt (val=2.726663)
[003/100] train_loss=2.001604 | val_loss=0.861840 | lr=0.0005
  ↳ Best model saved: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/CenterSpeedBottleneckCBAM_no_spatial20250821_215148_epoch_3.pt (val=0.861840)
[004/100] train_loss=0.714736 | val_loss=0.744105 | lr=0.0005
  ↳ Best model saved: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/CenterSpeedBottleneckCBAM_no_spatial20250821_215456_epoch_4.pt (val=0.744105)
[005/100] train_loss=0.620720 | val_loss=0.672962 | lr=0.0005
  ↳ Best model saved: /home/harry/ros2_ws/src/TinyCenterSpeed/src/pt/CenterSpeedBottleneckCBAM_no_s

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7a1027834b80>> (for post_run_cell), with arguments args (<ExecutionResult object at 7a10254a24a0, execution_count=4 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7a10254a1660, raw_cell="

best_val = float('inf')
train_hist, val_hist = [.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/harry/ros2_ws/src/TinyCenterSpeed/src/train/1_train_obj_free_lr_wandb.ipynb#W4sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe